## 1. Configuration

In [1]:
import os, re
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, pandas as pd, nibabel as nib, torch, torch.nn as nn, torch.nn.functional as F
from scipy import ndimage
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

BASE = r"C:\Users\FatmaH\dataset\PKG - RSNA-ASNR-MICCAI-BraTS-2021\RSNA-ASNR-MICCAI-BraTS-2021"
TRAIN_DIRS = [os.path.join(BASE,"BraTS2021_TrainingSet","TCGA-GBM"),
              os.path.join(BASE,"BraTS2021_TrainingSet","TCGA-LGG")]
VALID_DIRS = [os.path.join(BASE,"BraTS2021_ValidationSet","TCGA-GBM"),
              os.path.join(BASE,"BraTS2021_ValidationSet","TCGA-LGG")]

MAPPING_XLSX = r"C:\Users\FatmaH\dataset\BraTS2021_MappingToTCIA.xlsx"
CLINICAL_CSV = r"C:\Users\FatmaH\dataset\clinical\tcga_clinical.csv"
OUT_DIR      = r"C:\Users\FatmaH\dataset\Step2_features_final"
os.makedirs(OUT_DIR, exist_ok=True)

# ORIGINAL model (trained WITH TCGA -> in-domain). NOT the NOTCGA one.
CHECKPOINT   = "best_unet3d_brats2021_p128_bf32_rtx5090.pth"

# Preprocessed dir used for the original 80/20 split (to reproduce it exactly)
RAW_IMG_DIR  = r"C:\Users\FatmaH\dataset\preprocessed_brats2021-fat\images_raw"

PATCH_SIZE, BASE_FILTERS, NUM_CLASSES = 128, 32, 4
MIN_TCGA_WT_DICE = 0.80        # gate: refuse to predict the 76 if the model fails this

for p in TRAIN_DIRS+VALID_DIRS+[MAPPING_XLSX, CLINICAL_CSV, CHECKPOINT]:
    print(f"{'ok ' if (os.path.isdir(p) or os.path.isfile(p)) else 'MISSING'} {p}")
if "NOTCGA" in CHECKPOINT:
    print("\n*** This must be the ORIGINAL (with-TCGA) checkpoint, not NOTCGA. ***")

device: cpu
MISSING C:\Users\FatmaH\dataset\PKG - RSNA-ASNR-MICCAI-BraTS-2021\RSNA-ASNR-MICCAI-BraTS-2021/BraTS2021_TrainingSet/TCGA-GBM
MISSING C:\Users\FatmaH\dataset\PKG - RSNA-ASNR-MICCAI-BraTS-2021\RSNA-ASNR-MICCAI-BraTS-2021/BraTS2021_TrainingSet/TCGA-LGG
MISSING C:\Users\FatmaH\dataset\PKG - RSNA-ASNR-MICCAI-BraTS-2021\RSNA-ASNR-MICCAI-BraTS-2021/BraTS2021_ValidationSet/TCGA-GBM
MISSING C:\Users\FatmaH\dataset\PKG - RSNA-ASNR-MICCAI-BraTS-2021\RSNA-ASNR-MICCAI-BraTS-2021/BraTS2021_ValidationSet/TCGA-LGG
MISSING C:\Users\FatmaH\dataset\BraTS2021_MappingToTCIA.xlsx
MISSING C:\Users\FatmaH\dataset\clinical\tcga_clinical.csv
MISSING best_unet3d_brats2021_p128_bf32_rtx5090.pth


## 2. Build the three groups

- **expert** — TrainingSet TCGA, features from the annotation
- **validate** — the subset of those the original model held out (masks available → honest test)
- **predict** — ValidationSet TCGA, no masks, needs the model

In [ ]:
import openpyxl
from collections import Counter

def norm_tcga_id(x):
    s=str(x).strip().upper(); m=re.match(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})",s)
    return m.group(1) if m else s
def brats_id_from_folder(n):
    m=re.search(r"BraTS2021_\d+",n); return m.group(0) if m else None
def find_seg_file(p):
    for fn in os.listdir(p):
        low=fn.lower()
        if low.endswith((".nii",".nii.gz")) and ("_seg" in low or low.endswith("seg.nii.gz")):
            return os.path.join(p,fn)
    return None

# ---- mapping ----
brats_to_tcga, tcga_collection, tcga_mgmt = {}, {}, {}
wb=openpyxl.load_workbook(MAPPING_XLSX,data_only=True)
for r in list(wb["TCIA"].iter_rows(values_only=True))[1:]:
    if r[0] and "TCGA" in str(r[0]).upper() and r[4]:
        b=str(r[4]).strip(); t=norm_tcga_id(r[2])
        brats_to_tcga[b]=t; tcga_collection[t]=r[0]; tcga_mgmt[t]=r[7]

# ---- reproduce the ORIGINAL model's split to know what it held out ----
all_prep = sorted(f.replace(".npz","") for f in os.listdir(RAW_IMG_DIR) if f.endswith(".npz"))
orig_train, orig_val = train_test_split(all_prep, test_size=0.2, random_state=42)
orig_train, orig_val = set(orig_train), set(orig_val)
print(f"original split reproduced: {len(orig_train)} train / {len(orig_val)} val")

def scan(dirs):
    out=[]
    for d in dirs:
        if not os.path.isdir(d): continue
        for name in sorted(os.listdir(d)):
            p=os.path.join(d,name)
            if os.path.isdir(p): out.append((name,p))
    return out

expert, validate, predict = [], [], []
seen=set()
for name,p in scan(TRAIN_DIRS):
    b=brats_id_from_folder(name); t=brats_to_tcga.get(b) if b else None
    seg=find_seg_file(p)
    if t is None or t in seen or seg is None: continue
    seen.add(t)
    expert.append((name,p,t,seg))
    if b in orig_val:                      # held out by the original model -> honest test
        validate.append((name,p,t,seg))
for name,p in scan(VALID_DIRS):
    b=brats_id_from_folder(name); t=brats_to_tcga.get(b) if b else None
    if t is None or t in seen: continue
    seen.add(t); predict.append((name,p,t))

print(f"\nexpert   (TrainingSet, expert masks) : {len(expert)}")
print(f"validate (held out by orig model)   : {len(validate)}")
print(f"predict  (ValidationSet, no masks)  : {len(predict)}")
print(f"TOTAL                                : {len(expert)+len(predict)}")
print("\nby collection:", dict(Counter(tcga_collection.get(t,'?') for _,_,t,_ in expert)),
      dict(Counter(tcga_collection.get(t,'?') for _,_,t in predict)))

original split reproduced: 1000 train / 251 val

expert   (TrainingSet, expert masks) : 167
validate (held out by orig model)   : 33
predict  (ValidationSet, no masks)  : 76
TOTAL                                : 243

by collection: {'TCGA-GBM': 102, 'TCGA-LGG': 65} {'TCGA-GBM': 33, 'TCGA-LGG': 43}


## 3. Model, inference helpers, loader, features

In [ ]:
# ============================================================
# 3D U-Net
# ============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_channels),
            nn.LeakyReLU(inplace=True),

            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_channels),
            nn.LeakyReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ImprovedUNet3D(nn.Module):

    def __init__(
        self,
        in_channels=4,
        out_channels=4,
        base_filters=16,
        dropout=0.3
    ):

        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, base_filters)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = DoubleConv(base_filters, base_filters * 2)
        self.pool2 = nn.MaxPool3d(2)

        self.enc3 = DoubleConv(base_filters * 2, base_filters * 4)
        self.pool3 = nn.MaxPool3d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(
            base_filters * 4,
            base_filters * 8
        )

        self.dropout = nn.Dropout3d(dropout)

        # Decoder
        self.up3 = nn.ConvTranspose3d(
            base_filters * 8,
            base_filters * 4,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            base_filters * 8,
            base_filters * 4
        )

        self.up2 = nn.ConvTranspose3d(
            base_filters * 4,
            base_filters * 2,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            base_filters * 4,
            base_filters * 2
        )

        self.up1 = nn.ConvTranspose3d(
            base_filters * 2,
            base_filters,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            base_filters * 2,
            base_filters
        )

        self.final_conv = nn.Conv3d(
            base_filters,
            out_channels,
            kernel_size=1
        )

        # Deep supervision heads — lightweight 1x1x1 convs reading
        # straight off the decoder feature maps at 1/2 and 1/4
        # resolution, before the final upsampling stages.
        self.aux_head2 = nn.Conv3d(base_filters * 2, out_channels, kernel_size=1)  # from d2, 1/2 res
        self.aux_head3 = nn.Conv3d(base_filters * 4, out_channels, kernel_size=1)  # from d3, 1/4 res

        self._initialize_weights()

    def _initialize_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Conv3d):

                nn.init.kaiming_normal_(
                    m.weight,
                    nonlinearity="leaky_relu"
                )

                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.InstanceNorm3d):

                if m.weight is not None:
                    nn.init.constant_(m.weight, 1)

                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    @staticmethod
    def _pad(x, ref):

        diffZ = ref.size(2) - x.size(2)
        diffY = ref.size(3) - x.size(3)
        diffX = ref.size(4) - x.size(4)

        if diffZ or diffY or diffX:

            x = F.pad(
                x,
                [
                    diffX // 2,
                    diffX - diffX // 2,

                    diffY // 2,
                    diffY - diffY // 2,

                    diffZ // 2,
                    diffZ - diffZ // 2,
                ]
            )

        return x

    def forward(self, x, deep_supervision=False):

        e1 = self.enc1(x)

        e2 = self.enc2(self.pool1(e1))

        e3 = self.enc3(self.pool2(e2))

        b = self.dropout(
            self.bottleneck(
                self.pool3(e3)
            )
        )

        d3 = self.up3(b)
        d3 = self._pad(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self._pad(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self._pad(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        main_out = self.final_conv(d1)

        if deep_supervision and self.training:
            aux2 = self.aux_head2(d2)   # 1/2 resolution logits
            aux3 = self.aux_head3(d3)   # 1/4 resolution logits
            return main_out, aux2, aux3

        return main_out


# ============================================================
# Model
# ============================================================

model = ImprovedUNet3D(
    in_channels=4,
    out_channels=4,
    base_filters=16
).to(device)

model.eval()

with torch.no_grad():

    dummy = torch.randn(
        1,
        4,
        PATCH_SIZE,
        PATCH_SIZE,
        PATCH_SIZE
    ).to(device)

    output = model(dummy)

    print(f"\nInput shape  : {dummy.shape}")
    print(f"Output shape : {output.shape}")

del dummy, output

if torch.cuda.is_available():
    torch.cuda.empty_cache()

total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"\nTrainable parameters: {total_params:,}")

model.train()


Input shape  : torch.Size([1, 4, 128, 128, 128])
Output shape : torch.Size([1, 4, 128, 128, 128])

Trainable parameters: 1,402,300


ImprovedUNet3D(
  (enc1): DoubleConv(
    (block): Sequential(
      (0): Conv3d(4, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, bias=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (4): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, bias=False, track_running_stats=False)
      (5): LeakyReLU(negative_slope=0.01, inplace=True)
    )
  )
  (pool1): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): DoubleConv(
    (block): Sequential(
      (0): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, bias=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): Conv3d(32, 32, kernel_size=(3, 3, 

In [ ]:
model = ImprovedUNet3D(in_channels=4,out_channels=NUM_CLASSES,base_filters=BASE_FILTERS).to(device)
ck = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ck["model_state_dict"]); model.eval()
print(f"loaded {CHECKPOINT}  (epoch {ck.get('epoch','?')}, val Dice {ck.get('best_val_dice',float('nan')):.4f})")

loaded best_unet3d_brats2021_p128_bf32_rtx5090.pth  (epoch 37, val Dice 0.8771)


In [ ]:
import torch.nn.functional as F
from scipy import ndimage


def gaussian_importance_map(patch_size, sigma_scale=0.125):
    """Gaussian weighting so overlapping window edges blend smoothly
    instead of showing seam artifacts (nnU-Net-style)."""
    coords = np.arange(patch_size) - patch_size // 2
    sigma = patch_size * sigma_scale
    g1d = np.exp(-(coords ** 2) / (2 * sigma ** 2))
    g3d = np.einsum('i,j,k->ijk', g1d, g1d, g1d)
    g3d = g3d / g3d.max()
    return g3d.astype(np.float32)


def get_window_starts(dim_size, patch_size, stride):
    """Overlapping window start positions covering the full axis,
    including a final window flush with the far edge."""
    if dim_size <= patch_size:
        return [0]
    starts = list(range(0, dim_size - patch_size + 1, stride))
    if starts[-1] != dim_size - patch_size:
        starts.append(dim_size - patch_size)
    return starts


@torch.no_grad()
def sliding_window_inference(
    model,
    image,                # (4, H, W, D) torch tensor, already normalized
    patch_size=128,
    overlap=0.5,
    num_classes=4,
    device="cuda",
    use_tta=True
):
    """Returns averaged softmax probabilities, shape (num_classes, H, W, D)."""

    model.eval()

    _, H, W, D = image.shape
    stride = max(int(patch_size * (1 - overlap)), 1)

    weight_np = gaussian_importance_map(patch_size)
    weight = torch.from_numpy(weight_np).to(device)

    prob_sum = torch.zeros((num_classes, H, W, D), device=device)
    weight_sum = torch.zeros((H, W, D), device=device)

    xs = get_window_starts(H, patch_size, stride)
    ys = get_window_starts(W, patch_size, stride)
    zs = get_window_starts(D, patch_size, stride)

    flip_dims_list = [(2,), (3,), (4,)]  # H, W, D axes on a (1,4,P,P,P) tensor

    for x in xs:
        for y in ys:
            for z in zs:

                patch = image[:, x:x+patch_size, y:y+patch_size, z:z+patch_size]
                patch = patch.unsqueeze(0).to(device)

                probs = F.softmax(model(patch), dim=1)[0]

                if use_tta:
                    count = 1
                    for flip_dims in flip_dims_list:
                        flipped = torch.flip(patch, dims=flip_dims)
                        out_f = model(flipped)
                        out_f = torch.flip(out_f, dims=flip_dims)
                        probs = probs + F.softmax(out_f, dim=1)[0]
                        count += 1
                    probs = probs / count

                prob_sum[:, x:x+patch_size, y:y+patch_size, z:z+patch_size] += probs * weight
                weight_sum[x:x+patch_size, y:y+patch_size, z:z+patch_size] += weight

    prob_sum = prob_sum / weight_sum.unsqueeze(0).clamp(min=1e-8)
    return prob_sum


def keep_largest_component(pred_mask, num_classes=4):
    """Per class (excluding background), keep only the largest
    connected component. Removes small spurious false-positive blobs
    that hurt Dice without adding real clinical information."""

    cleaned = np.zeros_like(pred_mask)

    for c in range(1, num_classes):

        binary = (pred_mask == c)

        if binary.sum() == 0:
            continue

        labeled, n_components = ndimage.label(binary)

        if n_components == 0:
            continue

        sizes = ndimage.sum(binary, labeled, range(1, n_components + 1))
        largest_label = np.argmax(sizes) + 1

        cleaned[labeled == largest_label] = c

    return cleaned


def dice_score(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    if a.sum() == 0 and b.sum() == 0:
        return 1.0
    intersection = np.logical_and(a, b).sum()
    return (2.0 * intersection) / (a.sum() + b.sum() + 1e-8)


def compute_region_dice(pred, gt):
    """BraTS-standard region grouping.
    WT = whole tumor (all labels), TC = tumor core (NCR+ET), ET = enhancing tumor only.
    Assumes labels already remapped so ET=3."""

    return {
        "WT": dice_score(pred > 0, gt > 0),
        "TC": dice_score(np.isin(pred, [1, 3]), np.isin(gt, [1, 3])),
        "ET": dice_score(pred == 3, gt == 3),
    }

print("Sliding-window inference, TTA, and postprocessing utilities ready.")

Sliding-window inference, TTA, and postprocessing utilities ready.


In [ ]:
import nibabel as nib

MODALITY_TAGS = {
    "t1":    ["_t1.",    "_t1w.",   "t1."],
    "t1ce":  ["_t1ce.",  "_t1gd.",  "_t1c.",  "t1ce."],
    "t2":    ["_t2.",    "t2."],
    "flair": ["_flair.", "flair."],
}
CHANNEL_ORDER = ["t1", "t1ce", "t2", "flair"]

def find_modality_file(patient_path, tags):
    for fn in os.listdir(patient_path):
        low = fn.lower()
        if low.endswith((".nii", ".nii.gz")):
            for tag in tags:
                if tag in low:
                    return os.path.join(patient_path, fn)
    return None

def normalize(vol):
    """Min-max to [0,1] — MUST match the training preprocessing exactly."""
    vmin, vmax = vol.min(), vol.max()
    if vmax - vmin < 1e-8:
        return np.zeros_like(vol, dtype=np.float32)
    return ((vol - vmin) / (vmax - vmin)).astype(np.float32)

def load_tcga_patient(patient_path):
    channels = []
    for mod in CHANNEL_ORDER:
        fpath = find_modality_file(patient_path, MODALITY_TAGS[mod])
        if fpath is None:
            return None, f"missing_{mod}"
        vol = nib.load(fpath).get_fdata().astype(np.float32)
        channels.append(normalize(vol))          # <-- min-max, not z-score
    shapes = {c.shape for c in channels}
    if len(shapes) != 1:
        return None, f"shape_mismatch_{shapes}"
    return np.stack(channels, axis=0), "ok"

# sanity check: values must land in [0,1]
_test = predict[0][:3] if predict else (validate[0][:3] if validate else expert[0][:3])
_n, _p, _i = _test
_v, _s = load_tcga_patient(_p)
print("Test:", _n, "->", _s, "| shape:", None if _v is None else _v.shape)
if _v is not None:
    print(f"value range: [{_v.min():.3f}, {_v.max():.3f}]  (must be [0,1] to match training)")

Test: BraTS2021_00114 -> ok | shape: (4, 240, 240, 155)
value range: [0.000, 1.000]  (must be [0,1] to match training)


In [ ]:
from skimage.measure import marching_cubes, mesh_surface_area

def ce_rim_width_fixed(et_mask, spacing=(1.0,1.0,1.0)):
    """Mean enhancing-rim thickness (mm) via the slab approximation t ~= 2V/A.
    Replaces the original version, which returned 0 for every patient."""
    et = et_mask.astype(bool)
    if et.sum() < 10: return 0.0
    vv = float(np.prod(spacing)); vol = et.sum()*vv
    try:
        v,f,_,_ = marching_cubes(et.astype(np.float32), level=0.5, spacing=spacing)
        area = mesh_surface_area(v,f)
    except Exception:
        return 0.0
    return float(2.0*vol/area) if area > 0 else 0.0


def largest_wt_component(pred):
    """Keep only the largest connected whole-tumor component (labels preserved).
    Used for spatial-extent features so stray blobs cannot inflate size/hull."""
    wt = pred > 0
    lab, n = ndimage.label(wt)
    if n <= 1: return pred.copy()
    sizes = ndimage.sum(wt, lab, range(1, n+1))
    return np.where(lab == int(np.argmax(sizes))+1, pred, 0)


def extract_features(pred, spacing=(1.0,1.0,1.0)):
    vv  = float(np.prod(spacing))
    et  = pred == 3; ncr = pred == 1; ed = pred == 2
    wt  = et | ncr | ed; tc = et | ncr
    cc  = lambda m: float(m.sum()*vv/1000.0)

    f = {}
    # volumes
    f["et_volume"]=cc(et); f["ncr_volume"]=cc(ncr); f["ed_volume"]=cc(ed)
    f["wt_volume"]=cc(wt); f["tc_volume"]=cc(tc)
    f["residual_vol"]=np.nan                      # needs post-op scan

    # shape / boundary
    f["ce_rim_width"] = ce_rim_width_fixed(et, spacing)

    if et.sum() > 20:
        try:
            v,fc,_,_ = marching_cubes(et.astype(np.float32), level=0.5, spacing=spacing)
            surf = mesh_surface_area(v,fc); vol3 = et.sum()*vv
            f["surface_regularity"] = float(((36*np.pi*vol3**2)**(1/3))/(surf+1e-8))
        except Exception:
            f["surface_regularity"] = np.nan
    else:
        f["surface_regularity"] = np.nan

    wt_main = largest_wt_component(pred) > 0       # spatial features: main mass only
    if wt_main.sum() > 0:
        c = np.argwhere(wt_main)
        f["max_diameter"] = float(np.linalg.norm((c.max(0)-c.min(0))*np.array(spacing)))
    else:
        f["max_diameter"] = 0.0

    if wt_main.sum() > 20:
        try:
            from scipy.spatial import ConvexHull
            hull = ConvexHull(np.argwhere(wt_main)*np.array(spacing))
            f["solidity"] = min(float((wt_main.sum()*vv)/(hull.volume+1e-8)), 1.0)
        except Exception:
            f["solidity"] = np.nan
    else:
        f["solidity"] = np.nan

    if wt_main.sum() > 20:
        try:
            v,fc,_,_ = marching_cubes(wt_main.astype(np.float32), level=0.5, spacing=spacing)
            surf = mesh_surface_area(v,fc); vol3 = wt_main.sum()*vv
            f["sphericity"] = float((np.pi**(1/3))*(6*vol3)**(2/3)/(surf+1e-8))
        except Exception:
            f["sphericity"] = np.nan
    else:
        f["sphericity"] = np.nan

    # ratios / spatial
    f["et_wt_ratio"] = float(et.sum()/(wt.sum()+1e-8))
    ncr_et = float(ncr.sum()/(et.sum()+1e-8))
    f["ncr_et_ratio"] = ncr_et
    f["ncr_et_ratio_log"] = float(np.log1p(ncr_et))

    if wt.sum() > 0:
        cen = np.argwhere(wt).mean(0); mid = np.array(pred.shape)/2.0
        f["centroid_dev"] = float(abs(cen[0]-mid[0])*spacing[0])
    else:
        f["centroid_dev"] = 0.0

    return f

print("Feature functions ready (13 features; multifocal intentionally excluded).")

Feature functions ready (13 features; multifocal intentionally excluded).


## 4. Gate — honest TCGA Dice for the original model

Measured on the 33 patients it held out *and* that have expert masks. This decides whether the
model may be used for the 76 unlabelled patients.

In [ ]:
rows=[]
for name,path,tcga_id,seg_path in tqdm(validate, desc="validating original model on TCGA"):
    try:
        gt=nib.load(seg_path).get_fdata().astype(np.uint8); gt=gt.copy(); gt[gt==4]=3
        vol,st=load_tcga_patient(path)
        if vol is None: continue
        probs=sliding_window_inference(model, torch.from_numpy(vol).float(),
              patch_size=PATCH_SIZE, overlap=0.5, num_classes=NUM_CLASSES,
              device=device, use_tta=True)
        pred=probs.argmax(0).cpu().numpy().astype(np.uint8)
        pred=keep_largest_component(pred,num_classes=NUM_CLASSES)
        d=compute_region_dice(pred,gt); d["tcga_id"]=tcga_id
        d["ed_gt"]=(gt==2).sum()/1000; d["ed_pred"]=(pred==2).sum()/1000
        rows.append(d)
    except Exception as ex:
        print(" ",tcga_id,str(ex)[:60])

MODEL_OK=False
if rows:
    v=pd.DataFrame(rows); v.to_csv(os.path.join(OUT_DIR,"original_model_tcga_dice.csv"),index=False)
    print("\n"+"="*56)
    print(f"ORIGINAL model on TCGA (n={len(v)}, held out - honest)")
    print("="*56)
    for r in ["WT","TC","ET"]:
        print(f"  {r}: {v[r].mean():.4f}  (sd {v[r].std():.4f})")
    print(f"\n  edema volume: GT {v.ed_gt.mean():.1f} cm3 | predicted {v.ed_pred.mean():.1f} cm3")
    print(f"  ratio pred/GT: {v.ed_pred.mean()/max(v.ed_gt.mean(),1e-9):.2f}x   (NOTCGA model was 1.9x)")
    print(f"\n  compare NOTCGA model on TCGA: WT 0.4358 / TC 0.7191 / ET 0.7487")
    MODEL_OK = v["WT"].mean() >= MIN_TCGA_WT_DICE
    print("\n  ->", "PASS - safe to predict the 76 unlabelled patients" if MODEL_OK
                   else f"FAIL (WT < {MIN_TCGA_WT_DICE}) - use expert masks only")
else:
    print("no validation patients - check RAW_IMG_DIR and the split")

validating original model on TCGA:   0%|          | 0/33 [00:00<?, ?it/s]


ORIGINAL model on TCGA (n=33, held out - honest)
  WT: 0.9116  (sd 0.0606)
  TC: 0.7554  (sd 0.2484)
  ET: 0.6879  (sd 0.3191)

  edema volume: GT 52.1 cm3 | predicted 59.3 cm3
  ratio pred/GT: 1.14x   (NOTCGA model was 1.9x)

  compare NOTCGA model on TCGA: WT 0.4358 / TC 0.7191 / ET 0.7487

  -> PASS - safe to predict the 76 unlabelled patients


## 5. Extract — expert masks for 167, model for 76

In [ ]:
rows, failed = [], []

# ---- (a) expert masks ----
for name,path,tcga_id,seg_path in tqdm(expert, desc="expert masks"):
    try:
        seg=nib.load(seg_path).get_fdata().astype(np.uint8); seg=seg.copy(); seg[seg==4]=3
        f=extract_features(seg, spacing=(1.0,1.0,1.0))
        f.update(tcga_id=tcga_id, brats_folder=name, mask_source="expert",
                 collection=tcga_collection.get(tcga_id), mgmt_mapping=tcga_mgmt.get(tcga_id))
        f["low_enhancement"]=int(f["et_volume"]<0.5); rows.append(f)
    except Exception as ex: failed.append((tcga_id,"expert",str(ex)[:60]))

# ---- (b) model predictions, only if the gate passed ----
if MODEL_OK:
    for name,path,tcga_id in tqdm(predict, desc="model predictions"):
        try:
            vol,st=load_tcga_patient(path)
            if vol is None: failed.append((tcga_id,"model",st)); continue
            probs=sliding_window_inference(model, torch.from_numpy(vol).float(),
                  patch_size=PATCH_SIZE, overlap=0.5, num_classes=NUM_CLASSES,
                  device=device, use_tta=True)
            pred=probs.argmax(0).cpu().numpy().astype(np.uint8)
            pred=keep_largest_component(pred,num_classes=NUM_CLASSES)
            f=extract_features(pred, spacing=(1.0,1.0,1.0))
            f.update(tcga_id=tcga_id, brats_folder=name, mask_source="model",
                     collection=tcga_collection.get(tcga_id), mgmt_mapping=tcga_mgmt.get(tcga_id))
            f["low_enhancement"]=int(f["et_volume"]<0.5); rows.append(f)
        except Exception as ex: failed.append((tcga_id,"model",str(ex)[:60]))
else:
    print("gate failed - skipping the 76 model-predicted patients")

df=pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR,"tcga_features_hybrid.csv"),index=False)
print(f"\nextracted {len(df)}  (expert {int((df.mask_source=='expert').sum())}, "
      f"model {int((df.mask_source=='model').sum())})   failed {len(failed)}")

if len(df):
    print("\nmedian by source:")
    print(df.groupby("mask_source")[["wt_volume","ed_volume","et_volume","ce_rim_width"]]
            .median().round(2).to_string())
    print("\nIf the two rows differ wildly, restrict the analysis to mask_source=='expert'.")

expert masks:   0%|          | 0/167 [00:00<?, ?it/s]

gate failed - skipping the 76 model-predicted patients

extracted 167  (expert 167, model 0)   failed 0

median by source:
             wt_volume  ed_volume  et_volume  ce_rim_width
mask_source                                               
expert          105.85      51.85      11.99          3.44

If the two rows differ wildly, restrict the analysis to mask_source=='expert'.


In [ ]:
# ── Add the 76 model-predicted patients to the existing 167 ──
MODEL_OK = True          # Section 4 passed: WT 0.9116 >= 0.80

model_rows, failed_m = [], []
for name, path, tcga_id in tqdm(predict, desc="model predictions"):
    try:
        vol, st = load_tcga_patient(path)          # fixed min-max loader
        if vol is None:
            failed_m.append((tcga_id, st)); continue
        probs = sliding_window_inference(
            model, torch.from_numpy(vol).float(),
            patch_size=PATCH_SIZE, overlap=0.5,
            num_classes=NUM_CLASSES, device=device, use_tta=True)
        pred = probs.argmax(0).cpu().numpy().astype(np.uint8)
        pred = keep_largest_component(pred, num_classes=NUM_CLASSES)

        f = extract_features(pred, spacing=(1.0,1.0,1.0))
        f.update(tcga_id=tcga_id, brats_folder=name, mask_source="model",
                 collection=tcga_collection.get(tcga_id),
                 mgmt_mapping=tcga_mgmt.get(tcga_id))
        f["low_enhancement"] = int(f["et_volume"] < 0.5)
        model_rows.append(f)
    except Exception as ex:
        failed_m.append((tcga_id, str(ex)[:60]))

df = pd.concat([df, pd.DataFrame(model_rows)], ignore_index=True)
df.to_csv(os.path.join(OUT_DIR, "tcga_features_hybrid.csv"), index=False)

print(f"\ntotal {len(df)}  (expert {int((df.mask_source=='expert').sum())}, "
      f"model {int((df.mask_source=='model').sum())})   failed {len(failed_m)}")
print("\nmedian by source:")
print(df.groupby("mask_source")[["wt_volume","ed_volume","et_volume","ce_rim_width"]]
        .median().round(2).to_string())

model predictions:   0%|          | 0/76 [00:00<?, ?it/s]


total 243  (expert 167, model 76)   failed 0

median by source:
             wt_volume  ed_volume  et_volume  ce_rim_width
mask_source                                               
expert          105.85      51.85      11.99          3.44
model            90.54      50.02      10.61          3.63


## 6. Merge with clinical

In [ ]:
cli=pd.read_csv(CLINICAL_CSV); cli["tcga_id"]=cli.tcga_id.map(norm_tcga_id)
df["tcga_id"]=df.tcga_id.map(norm_tcga_id)
merged=df.merge(cli[["tcga_id","os_days","os_event","subtype"]],on="tcga_id",how="inner")
merged.to_csv(os.path.join(OUT_DIR,"tcga_features_with_clinical.csv"),index=False)

SUB=["Astrocytoma","Oligodendroglioma","Glioblastoma"]
print(f"features {len(df)} | clinical {len(cli)} | MATCHED {len(merged)}\n")
t=(merged.groupby("subtype").agg(n=("os_event","size"),events=("os_event","sum"),
    median_OS=("os_days","median")).reindex(SUB))
print(t.to_string())
print(f"\nPOOLED n={len(merged)}, events={int(merged.os_event.sum())}")
print("\nby mask source:")
print(merged.groupby("mask_source").agg(n=("os_event","size"),events=("os_event","sum")).to_string())
print("\nfeasibility:")
for st in SUB:
    ev=0 if st not in t.index or pd.isna(t.loc[st,"events"]) else int(t.loc[st,"events"])
    print(f"  {st:20} events={ev:>3} ->","OK" if ev>=40 else "WORKABLE" if ev>=25
          else "THIN" if ev>=12 else "TOO FEW")
print(f"\n~{int(merged.os_event.sum())//10} features max at 10 events per variable.")

features 243 | clinical 901 | MATCHED 213

                     n  events  median_OS
subtype                                  
Astrocytoma         61      11    517.480
Oligodendroglioma   27       5    432.248
Glioblastoma       125      93    340.928

POOLED n=213, events=109

by mask source:
               n  events
mask_source             
expert       143      70
model         70      39

feasibility:
  Astrocytoma          events= 11 -> TOO FEW
  Oligodendroglioma    events=  5 -> TOO FEW
  Glioblastoma         events= 93 -> OK

~10 features max at 10 events per variable.
